## Phase 6 : Génération de la soumission Kaggle

On applique au fichier `data/test.csv` exactement le même preprocessing que sur le train (imputation `intelligent_imputation`, encodage ordinal, one-hot avec alignement strict des colonnes sur `X_train.columns` via `reindex`), puis on prédit avec le modèle retenu en Phase 4. Les prédictions sont retransformées de l'espace `log1p` vers les dollars via `np.expm1` avant écriture dans `submission.csv`.

In [ ]:
import pandas as pd
import numpy as np

# 1. Sélection automatique du meilleur pipeline
# On rassemble d'abord tous nos pipelines entraînés dans un dictionnaire
all_trained_pipelines = {
    'Random Forest': pipelines['Random Forest'],
    'AdaBoost': pipelines['AdaBoost'],
    'XGBoost': pipelines['XGBoost'],
    'Stacking': pipelines['Stacking'],
    'Lasso': new_pipelines['Lasso'],
    'Ridge': new_pipelines['Ridge'],
    'ElasticNet': new_pipelines['ElasticNet'],
    'XGBoost Optimisé': grid_search.best_estimator_  # Le champion issu du GridSearchCV (Phase 4.5)
}

best_model_name = min(results_rmse, key=results_rmse.get)
best_pipeline = all_trained_pipelines[best_model_name]

print(f"Meilleur modèle retenu pour Kaggle : {best_model_name} (RMSE log = {results_rmse[best_model_name]:.4f})")

# 2. Chargement du jeu de test Kaggle
df_test = pd.read_csv('./data/test.csv', sep=',')
test_ids = df_test['Id']
print(f"Dimensions du test Kaggle brutes : {df_test.shape}")

# 3. Application du nettoyage "Métier" initial
df_test_clean = intelligent_imputation(df_test)

# 4. Prédiction directe
preds_log = best_pipeline.predict(df_test_clean)

# 5. On retransforme les prédictions (de l'espace log vers les dollars $)
preds_dollars = np.expm1(preds_log)

# 6. Sauvegarde de la soumission Kaggle
submission = pd.DataFrame({'Id': test_ids, 'SalePrice': preds_dollars})
submission.to_csv('submission.csv', index=False)

print(f"\nSUCCÈS ! Fichier 'submission.csv' généré ({len(submission)} lignes).")
display(submission.head())

print("\nStatistiques des prix prédits ($) :")
display(submission['SalePrice'].describe().to_frame().T)

TODO : ajouter une "histoire" en phase 1 pour synthétiser notre ML-CANVAS puis enchaîner avec l'EDA
Avoir peut-être plusieurs notes book (conseil du prof):
- 1 NB pour ML CANVAS et EDA --> EXPLIQUER POURQUOI ON UTILISE LE RMSLE est important et commenter l'EDA
- 1 NB par modèle si nécessaire --> BIEN EXPLIQUER LES MODELES QUE L'ON FAIT ET POURQUOI (TESTER TOUS LES MODELES VUS EN COURS)
- 1 NB pour la soumission
1- 1 POINT  (sur 6) PAR PHASE DE CRISP --> donc on doit être clean

A chaque fin de phase (CRISP) avoir un suivi permettant l'introduction de la phase suivante: ML CANVAS (histoire) -> EDA (raconté l'EDA) -> préparation données (pas trop de texte, mais expliquer pourquoi) -> évaluation (dissertation de qu'est-ce qu'un bon modèle, tous modèles ont des limites d'utilisation)
lors de l'analyse des modèles AVOIR UN GRAPHE avec valeur prédite et valeur réelle pour déterminer ou notre modèle est performant (les points les plus éloignés ne sont pas possibles à être déterminé)